# 02. Xe II: сечения выбранных линий и константы скоростей

* **σ возбуждения** верхнего уровня u из уровня i — Wang et al (2019), CrossSectionsIon.xlsx
  (листы gs->6p, 6s->6p, 5d->6p; i = 1, 2 — 5p⁵ ²P₃/₂, ²P₁/₂; i ≥ 4 — 6s, 5d);
* **ветвление** BR = A_line/ΣA — ноутбук 01 (`xe2_A_branching.csv`); для каждой линии
  выбран вариант «только DBSR» или «A из NIST, где есть» (`nb.XE2_LINES`, поле `br`);
* **σ линии** = BR · σ(i → u) для всех i, для которых у Ванга есть данные;
* **константы скоростей** ⟨σv⟩(Te) — максвелловское распределение, сетка `TE` (первая ячейка):
  для возбуждения уровня (k(i→u)) и для линии (BR·k(i→u)).

Каскады в верхние уровни здесь не учитываются (это часть модели CRM).

**Выход** (в папке `xe2_wang`): `xe2_line_sigma.xlsx` (формат Ванга: пары столбцов
Energy(eV) / Sigma(1E-16 cm^2), над столбцом σ — метка «i->u (λ)»), `xe2_k_excitation.xlsx`
и `xe2_k_line.xlsx` (+ `.csv`): строки — Te, столбцы — переходы.

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
REPO    = Path("/data/pydbsr_calc/DBSR_src")          # клон репозитория pydbsr
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = Path("/data/pydbsr_calc/data")              # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = Path("/data/pydbsr_calc/runs/crm")          # результаты расчётов (большой диск)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

In [ ]:
ref = reference.read_xlsx(XLSX)
W = RUNS / "xe2_wang"
tab = {(int(r["upper"]), int(r["lower"])): r for r in nb.read_csv(W / "xe2_A_branching.csv")}
col = {"dbsr": "branching", "hybrid": "branching_hybrid"}      # см. nb.XE2_LINES, поле br
lines = [dict(x, BR=float(tab[(x["upper"], x["lower"])][col[x["br"]]])) for x in nb.XE2_LINES]
for x in lines:
    init = sorted(i for (i, j) in ref.sigma if j == x["upper"])
    print(f"{x['wl']:8.3f}  {x['upper']}->{x['lower']}  BR = {x['BR']:.4f} ({x['br']})  σ у Ванга из уровней: {init}")

In [ ]:
sig, lab, sheet = {}, {}, {}
for x in lines:
    u = x["upper"]
    for (i, j), (E, s) in ref.sigma.items():
        if j == u:
            key = (i, u, x["wl"])
            sig[key] = (E, x["BR"] * s)
            lab[key] = f"{i}->{u} ({x['wl']:.3f})"
            sheet[key] = ref.sheet[(i, j)]
out = reference.write_xlsx(W / "xe2_line_sigma.xlsx", ref.levels, sig, sheet_of=lambda k: sheet[k], labels=lab)
print(len(sig), "сечений линий ->", out)

## Сечения линий из основного состояния и из ²P₁/₂

In [ ]:
fig, axs = plt.subplots(1, len(lines), figsize=(4 * len(lines), 3.5), squeeze=False)
for ax, x in zip(axs[0], lines):
    for i in (1, 2):
        if (i, x["upper"], x["wl"]) in sig:
            E, s = sig[(i, x["upper"], x["wl"])]
            ax.plot(E, s / 1e-16, lw=0.8, label=f"из {i}: {ref.label(i)}")
    ax.set_xscale("log"); ax.set_title(f"{x['wl']} нм ({x['upper']}→{x['lower']}), BR = {x['BR']:.3f}", fontsize=9)
    ax.set_xlabel("E, эВ"); ax.set_ylabel("σ линии, 10⁻¹⁶ см²")
axs[0][0].legend(fontsize=7); fig.tight_layout(); plt.show()

## Константы скоростей ⟨σv⟩(Te)

Ниже первой табличной точки σ держится равным первому значению (для иона σ конечно на
пороге), выше последней (109 эВ) — спадает как 1/E (`rates.rate_from_sigma`). При Te ≤ 20 эВ
вклад E > 109 эВ мал; проверьте по `tail="dipole"`, если нужно.

In [ ]:
TEa = np.array(TE)
k_exc, k_line, lab_e, lab_l = {}, {}, {}, {}
for x in lines:
    u = x["upper"]
    for (i, j), (E, s) in ref.sigma.items():
        if j != u:
            continue
        thr = (ref.level(u).energy_cm - ref.level(i).energy_cm) / nb.CM_PER_EV
        k = rates.rates_on_grid(E, s, TEa, thr)
        k_exc[(i, u)] = k
        lab_e[(i, u)] = f"{i}->{u}"
        k_line[(i, u, x["wl"])] = x["BR"] * k
        lab_l[(i, u, x["wl"])] = f"{i}->{u} ({x['wl']:.3f})"
reference.write_rates_xlsx(W / "xe2_k_excitation.xlsx", TEa, k_exc, lab_e,
                           note="<sigma v> (cm3/s), excitation i->u, sigma: Wang et al 2019 (CrossSectionsIon.xlsx), Maxwellian")
reference.write_rates_xlsx(W / "xe2_k_line.xlsx", TEa, k_line, lab_l,
                           note="BR * <sigma v> (cm3/s), line from u; BR: DBSR (notebook 01); sigma: Wang et al 2019")
print("->", W / "xe2_k_excitation.xlsx", W / "xe2_k_line.xlsx")
for x in lines:
    k = k_line[(1, x["upper"], x["wl"])]
    print(f"{x['wl']:8.3f} из 1: " + "  ".join(f"{t:g} эВ: {v:.2e}" for t, v in zip(TEa, k)))